In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [3]:
# Load dataset gốc để lấy scaler
df = pd.read_csv("D:\\Hoang's work\\PatchTST\\PatchTST_supervised\\dataset\\VN_Energy.csv")
ot_values = df["MB_MW"].values.reshape(-1, 1)

print(ot_values.shape)
print(ot_values[:5])

(38688, 1)
[[8776.2 ]
 [8762.6 ]
 [8740.15]
 [8735.05]
 [8609.7 ]]


In [18]:
df.head()

,date,MB_MW,MT_MW,MN_MW,HT_MW
0,2021-01-01 00:00:00,8776.20,1705.4,10259.40,20741.00
1,2021-01-01 01:00:00,8762.60,1619.4,10020.00,20402.00
2,2021-01-01 02:00:00,8740.15,1661.6,9870.80,20272.55
3,2021-01-01 03:00:00,8735.05,1674.8,9718.55,20128.40
4,2021-01-01 04:00:00,8609.70,1663.8,9453.05,19726.55


In [15]:
# Fit scaler như trong data_loader của repo
scaler = StandardScaler()
scaler.fit(ot_values)

# Load prediction (chuẩn hóa)
preds = np.load("D:\\Hoang's work\\PatchTST\\PatchTST_supervised\\results\VN_Energy_336_336_PatchTST_VN_Energy_ftM_sl336_ll48_pl336_dm32_nh4_el3_dl1_df256_fc1_ebtimeF_dtTrue_Exp_0\\real_prediction.npy")

# Chuyển ngược về giá trị gốc
preds_real = scaler.inverse_transform(preds.reshape(-1, 1)).reshape(preds.shape)

print(preds_real.shape)
print(preds_real[:1, :100, 3])  # 100 giá trị đầu tiên của col 1

(1, 336, 4)
[[16148.06   15272.493  14683.849  14468.432  14282.763  14143.878
  14320.781  14224.529  15297.198  16263.469  16638.377  16625.041
  16411.094  16337.381  17443.041  17499.459  18106.441  18251.328
  18363.557  18422.674  18150.53   18041.705  17473.93   16588.373
  15546.503  14583.411  14065.731  14208.217  14096.86   14213.855
  14741.143  15496.715  17526.97   18873.453  19699.105  19777.514
  19433.871  19754.174  20731.44   21321.227  21485.553  21211.723
  20866.328  20787.299  20472.867  20139.334  19468.635  18312.445
  17334.992  16407.242  16042.466  15922.047  15674.588  15911.816
  16296.868  16935.004  18713.258  20179.125  20945.15   20678.854
  20100.523  20519.975  21430.262  22108.604  22093.62   21698.523
  21362.393  21022.371  20514.232  20161.088  19653.932  18755.508
  17575.943  16578.191  16131.303  15923.678  15759.174  15892.299
  16305.4375 17093.352  18745.965  20178.02   20857.727  20852.402
  20266.643  20568.291  21522.785  22382.465  2234

In [ ]:
# gộp tất cả batch, stack theo time
preds_flat = preds_real.reshape(-1, preds_real.shape[-1])  # (batch*pred_len, features)

print(preds_flat.shape)

(336, 4)


In [19]:
# Lấy số điểm dự đoán
n_preds = preds_flat.shape[0]

In [21]:
# Tạo datetime index mới nối tiếp từ dataset gốc
last_date = pd.to_datetime(df["date"].iloc[-1])
future_dates = pd.date_range(start=last_date + pd.Timedelta(hours=1), 
                             periods=n_preds, freq="H")

C:\Users\Hi\AppData\Local\Temp\ipykernel_143908\2023215473.py:3: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  future_dates = pd.date_range(start=last_date + pd.Timedelta(hours=1),


In [22]:
# File 2: Prediction DataFrame
df_pred = pd.DataFrame(preds_flat, columns=df.columns[1:])  # bỏ cột date
df_pred.insert(0, "date", future_dates)

# File 1: Append vào dataset gốc
df_extended = pd.concat([df, df_pred], ignore_index=True)

# Save ra 2 file
df_pred.to_csv("predictions_only.csv", index=False)
df_extended.to_csv("dataset_with_predictions.csv", index=False)